# SAC Agent with Unity ML-Agents Environment

This notebook demonstrates how to train a **Soft Actor-Critic (SAC)** agent from TensorAeroSpace with a Unity ML-Agents environment.

## Overview

- **Algorithm**: Soft Actor-Critic (SAC) — off-policy actor-critic with entropy maximization
- **Environment**: Unity ML-Agents continuous control environment
- **Features**: Automatic entropy tuning, Gaussian policy, experience replay

## Prerequisites

1. Unity ML-Agents environment (Editor or standalone build)
2. `mlagents` Python package
3. TensorAeroSpace library

**Unity Environment Repository:** [TensorAeroSpace/UnityAirplaneEnvironment](https://github.com/TensorAeroSpace/UnityAirplaneEnvironment)

For Unity environment setup, see the [Unity Environment Guide](https://tensoraerospace.github.io/TensorAeroSpace/guide/unity_env/).


## 1. Installation

Install the required ML-Agents package if not already installed:


In [ ]:
# Install ML-Agents (uncomment and run if needed)
# !pip install mlagents==1.1.0


## 2. Imports

Import the necessary libraries:


In [ ]:
import gymnasium as gym
import numpy as np

from tensoraerospace.agent import SAC

In [2]:
from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.envs.unity_gym_env import UnityToGymWrapper

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 3. Gymnasium Wrapper for Unity Environment

Unity ML-Agents uses the old `gym` API. We need a wrapper to convert it to the modern `gymnasium` API
that TensorAeroSpace expects.

The wrapper handles:
- `reset()`: Returns `(observation, info)` instead of just `observation`
- `step()`: Returns 5-tuple `(obs, reward, terminated, truncated, info)` instead of 4-tuple


In [ ]:
class UnityToGymnasiumWrapper(gym.Wrapper):
    """Wrapper to convert Unity ML-Agents gym environment to Gymnasium API.
    
    This wrapper adapts the old gym API (used by mlagents_envs) to the modern
    Gymnasium API expected by TensorAeroSpace agents.
    
    Args:
        env: Unity environment wrapped with UnityToGymWrapper
    """

    def __init__(self, env):
        super().__init__(env)
        self.env = env

    def reset(self, *, seed=None, options=None):
        """Reset the environment and return initial observation.
        
        Returns:
            tuple: (observation, info) following Gymnasium API
        """
        obs = self.env.reset()
        return obs, {}

    def step(self, action):
        """Execute one step in the environment.
        
        Args:
            action: Action to execute (continuous for SAC)
            
        Returns:
            tuple: (observation, reward, terminated, truncated, info)
                   following Gymnasium API (5-tuple instead of 4-tuple)
        """
        # Old gym API: (obs, reward, done, info)
        result = self.env.step(action)
        obs, reward, done, info = result
        # New gymnasium API: (obs, reward, terminated, truncated, info)
        # Unity doesn't distinguish terminated/truncated, so truncated=False
        return obs, reward, done, False, info

    def close(self):
        """Close the Unity environment."""
        self.env.close()

## 4. Connect to Unity Environment

Connect to the Unity environment. You have two options:

### Option A: Unity Editor (Development)
```python
unity_env = UnityEnvironment(None)  # Connects to running Unity Editor
```

### Option B: Standalone Build (Production/Training)
```python
unity_env = UnityEnvironment("/path/to/build.x86_64")  # Linux
unity_env = UnityEnvironment("C:\\path\\to\\build.exe")  # Windows
```

**Note**: Press Play in Unity Editor after running this cell if using Option A.


In [ ]:
# Connect to Unity environment
# Option A: Unity Editor (set file_name=None)
# Option B: Standalone build (provide path to executable)

unity_env = UnityEnvironment(
    file_name=None,  # None for Editor, or path to build
    log_folder="./logs/",
    additional_args=["-logfile", "unity.log"]
)

# Wrap for gym/gymnasium compatibility
gym_env = UnityToGymWrapper(unity_env, uint8_visual=True)
env = UnityToGymnasiumWrapper(gym_env)

print(f"Action space: {env.action_space}")
print(f"Observation space: {env.observation_space}")

[WARNING] uint8_visual was set to true, but visual observations are not in use. This setting will not have any effect.
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.


/root/.cache/pypoetry/virtualenvs/tensoraerospace-9TtSrW0h-py3.10/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


## 5. Configure SAC Agent

Create and configure the SAC agent with hyperparameters optimized for continuous control:

| Parameter | Value | Description |
|-----------|-------|-------------|
| `batch_size` | 256 | Mini-batch size for updates |
| `memory_capacity` | 1,000,000 | Replay buffer size |
| `lr` | 3e-4 | Learning rate for critic networks |
| `policy_lr` | 3e-4 | Learning rate for actor network |
| `gamma` | 0.99 | Discount factor |
| `tau` | 0.005 | Soft update coefficient |
| `automatic_entropy_tuning` | True | Auto-adjust exploration |
| `hidden_size` | 256 | Hidden layer neurons |


In [ ]:
agent = SAC(
    env=env,
    # Training dynamics
    updates_per_step=1,
    batch_size=256,
    memory_capacity=1_000_000,
    
    # Learning rates
    lr=3e-4,          # Critic learning rate
    policy_lr=3e-4,   # Actor learning rate
    
    # RL hyperparameters
    gamma=0.99,       # Discount factor
    tau=0.005,        # Soft update coefficient
    alpha=0.2,        # Initial entropy coefficient (auto-tuned if enabled)
    
    # Policy configuration
    policy_type="Gaussian",
    target_update_interval=1,
    automatic_entropy_tuning=True,
    
    # Network architecture
    hidden_size=256,
    
    # Device and logging
    device="cuda",  # Use "cpu" if no GPU available
    verbose_histogram=False,
    seed=42,
)

print("SAC agent initialized successfully!")
print(f"Device: {agent.device}")

## 6. Training

Train the agent. Training logs are saved to TensorBoard:

```bash
tensorboard --logdir runs/
```

**Tips for Unity environments**:
- Start with fewer episodes to verify the connection
- Monitor Unity console for any environment errors
- Ensure Unity scene stays in Play mode during training


In [ ]:
# Train the agent
NUM_EPISODES = 1000

agent.train(num_episodes=NUM_EPISODES)

  2%|▏         | 19/1000 [01:07<53:30,  3.27s/it]  

## 7. Evaluation

Evaluate the trained agent over multiple episodes:


In [ ]:
def evaluate_agent(agent, env, num_episodes=5):
    """Evaluate trained agent over multiple episodes.
    
    Args:
        agent: Trained SAC agent
        env: Environment to evaluate on
        num_episodes: Number of evaluation episodes
        
    Returns:
        tuple: (mean_reward, std_reward, all_rewards)
    """
    rewards = []
    
    for episode in range(num_episodes):
        state, info = env.reset()
        done = False
        total_reward = 0
        
        while not done:
            # Use evaluate=True for deterministic action selection
            action = agent.select_action(state, evaluate=True)
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
        
        rewards.append(total_reward)
        print(f"Episode {episode + 1}: Reward = {total_reward:.2f}")
    
    mean_reward = np.mean(rewards)
    std_reward = np.std(rewards)
    
    print(f"\n{'='*40}")
    print(f"Mean Reward: {mean_reward:.2f} ± {std_reward:.2f}")
    print(f"{'='*40}")
    
    return mean_reward, std_reward, rewards


# Run evaluation
mean_reward, std_reward, all_rewards = evaluate_agent(agent, env, num_episodes=5)

83.45379334688187


## 8. Save and Load Model

Save the trained model for later use:


In [ ]:
# Save the trained model
agent.save(path="./checkpoints")

{'gamma': 0.99, 'tau': 0.005, 'alpha': 0.050813738256692886, 'verbose_histogram': False, 'memory_capacity': 1000000, 'policy_type': 'Gaussian', 'updates_per_step': 1, 'target_update_interval': 1, 'batch_size': 256, 'automatic_entropy_tuning': True, 'device': 'cuda', 'lr': 0.0003}


In [ ]:
# Load a pretrained model (example)
# loaded_agent = SAC.from_pretrained("./checkpoints/Nov24_11-01-27_SAC")


## 9. Cleanup

Close the Unity environment when finished:


In [ ]:
# Close the environment
env.close()
print("Environment closed successfully.")


## Troubleshooting

### Common Issues

| Issue | Solution |
|-------|----------|
| `Port 5004 is busy` | Close other Unity instances or change port |
| `allow_multiple_obs` warning | Set `allow_multiple_obs=True` in wrapper or ignore |
| Connection timeout | Start Unity scene before running Python script |
| CUDA out of memory | Reduce `batch_size` or `hidden_size` |

### Typical Connection Log

```
[INFO] Listening on port 5004...
[INFO] Connected to Unity environment with package version X.X.X
[INFO] Connected new brain: BehaviorName?team=0
```

## References

- [SAC Paper](https://arxiv.org/abs/1801.01290)
- [Unity ML-Agents Documentation](https://unity-technologies.github.io/ml-agents/)
- [TensorAeroSpace SAC Documentation](https://tensoraerospace.github.io/TensorAeroSpace/agent/sac/)
